In [1]:
import json
import pandas as pd
from collections import defaultdict, Counter
import re

**REVIEWING CATEGORIES THAT WERE FREQUENTLY MISCLASSFIED**

In [ ]:
OUTPUT_CSV = "SCOTBESS_DATASET_MASKED.csv"
ENTITIES_COL = "llm_entities_json"

ENTITY_CATEGORIES = [
    "APPLICATION_CODE",
    "PERSON",
    "ORGANIZATION",
    "PLACE"]

masked_df = pd.read_csv(OUTPUT_CSV)

# value -> Counter({category: number_of_rows_detected})
counts_by_value = defaultdict(Counter)

parse_errors = []

for row_index, raw_output in masked_df[ENTITIES_COL].items():

    if pd.isna(raw_output) or not str(raw_output).strip():
        continue

    try:
        entities = (
            json.loads(raw_output)
            if isinstance(raw_output, str)
            else raw_output)

        for category in ENTITY_CATEGORIES:

            # Do not count the same value twice within one document
            row_values = {
                str(value).strip()
                for value in entities.get(category, [])
                if str(value).strip()}

            for value in row_values:
                counts_by_value[value][category] += 1

    except (json.JSONDecodeError, TypeError, AttributeError) as error:
        parse_errors.append({
            "row_index": row_index,
            "error": str(error),
            "raw_output": raw_output})

print(f"Unique candidate values: {len(counts_by_value)}")
print(f"Rows with parsing errors: {len(parse_errors)}")

Unique candidate values: 2212
Rows with parsing errors: 0


In [ ]:
review_rows = []

for value, category_counts in counts_by_value.items():

    categories = sorted(category_counts.keys())

    review_rows.append({
        "value": value,
        "detection_frequency": sum(category_counts.values()),
        "categories": " | ".join(categories),
        "category_frequencies": " | ".join(
            f"{category}: {category_counts[category]}"
            for category in categories),
        "category_conflict": len(categories) > 1})

candidate_review_df = (
    pd.DataFrame(review_rows).sort_values(
        ["category_conflict", "detection_frequency", "value"],
        ascending=[False, False, True]).reset_index(drop=True))

In [ ]:
conflicts_df = candidate_review_df[candidate_review_df["category_conflict"]].copy()

conflicts_df = candidate_review_df[candidate_review_df["category_conflict"]].copy()
conflicts_df["final_category"] = ""
#exporting for manual review/resolving the overlapping categories
#conflicts_df.to_excel("masking_category_conflicts_review.xlsx",index=False)

display(conflicts_df)

,value,detection_frequency,categories,category_frequencies,category_conflict,final_category
0,Rothienorman,79,PERSON | PLACE,PERSON: 2 | PLACE: 77,True,
1,SSEN,61,ORGANIZATION | PERSON,ORGANIZATION: 60 | PERSON: 1,True,
2,Mey,43,PERSON | PLACE,PERSON: 1 | PLACE: 42,True,
3,Highland Council,28,ORGANIZATION | PERSON | PLACE,ORGANIZATION: 25 | PERSON: 1 | PLACE: 2,True,
4,East Lothian Council,23,ORGANIZATION | PERSON | PLACE,ORGANIZATION: 14 | PERSON: 2 | PLACE: 7,True,
5,Scottish Fire and Rescue Service,21,ORGANIZATION | PERSON,ORGANIZATION: 20 | PERSON: 1,True,
6,Pegasus Group,19,ORGANIZATION | PERSON,ORGANIZATION: 18 | PERSON: 1,True,
7,Cockenzie Storage Limited,15,ORGANIZATION | PERSON,ORGANIZATION: 14 | PERSON: 1,True,
8,Hollandmey,15,ORGANIZATION | PLACE,ORGANIZATION: 1 | PLACE: 14,True,
9,Danestone,11,PERSON | PLACE,PERSON: 1 | PLACE: 10,True,


In [5]:
#df with manually resolved overlapping categories
REVIEW_XLSX = "masking_category_conflicts_review.xlsx"

resolved_conflicts = pd.read_excel(REVIEW_XLSX)
resolved_conflicts["final_category"] = (resolved_conflicts["final_category"].fillna("").str.strip().str.upper())
display(resolved_conflicts[["value", "categories", "final_category"]])

,value,categories,final_category
0,Rothienorman,PERSON | PLACE,PLACE
1,SSEN,ORGANIZATION | PERSON,ORGANIZATION
2,Mey,PERSON | PLACE,PLACE
3,Highland Council,ORGANIZATION | PERSON | PLACE,ORGANIZATION
4,East Lothian Council,ORGANIZATION | PERSON | PLACE,ORGANIZATION
5,Scottish Fire and Rescue Service,ORGANIZATION | PERSON,ORGANIZATION
6,Pegasus Group,ORGANIZATION | PERSON,ORGANIZATION
7,Cockenzie Storage Limited,ORGANIZATION | PERSON,ORGANIZATION
8,Hollandmey,ORGANIZATION | PLACE,PLACE
9,Danestone,PERSON | PLACE,PLACE


In [6]:
conflict_resolutions = dict(zip(resolved_conflicts["value"], resolved_conflicts["final_category"]))

In [ ]:
ENTITY_CATEGORIES = [
    "APPLICATION_CODE",
    "PERSON",
    "ORGANIZATION",
    "PLACE"]
#the llm struggled with detecting the entities, eg. it masked "Aberdeen" in one representation but not the next one
#so the above defined categories will be converted into sets and detected globally, to increase recall
#other categries (email, phone, link) will be resolved on project (row) level, as they are unambiguous

candidate_sets = {
    category: set()
    for category in ENTITY_CATEGORIES}

for _, row in candidate_review_df.iterrows():
    value = row["value"]

    if bool(row["category_conflict"]):
        final_category = conflict_resolutions.get(value, "")
    else:
        #non-conflicting candidates keep their original category
        final_category = row["categories"]

    if final_category == "REJECT":
        continue

    if final_category not in candidate_sets:
        raise ValueError(
            f"Missing or invalid category for {value!r}: "
            f"{final_category!r}")

    candidate_sets[final_category].add(value)

In [8]:
APPLICATION_CODE_SET = candidate_sets["APPLICATION_CODE"]
PERSON_SET = candidate_sets["PERSON"]
ORGANIZATION_SET = candidate_sets["ORGANIZATION"]
PLACE_SET = candidate_sets["PLACE"]

for category, values in candidate_sets.items():
    print(f"{category}: {len(values)} unique candidates")

APPLICATION_CODE: 149 unique candidates
PERSON: 211 unique candidates
ORGANIZATION: 345 unique candidates
PLACE: 1507 unique candidates


In [9]:
#for inspection
for category, values in candidate_sets.items():
    print(f"\n--- {category} ---")

    for value in sorted(values, key=str.casefold):
        print(repr(value))


--- APPLICATION_CODE ---
'00004838'
'100655885- 001'
'100711077-001'
'13/0479/FUL'
'15/03392/FUL'
'19/01861/S36'
'19/05385/FUL'
'20/02823/PAN'
'20/9004/PREAPP'
'2023/0378/TP'
'2024/0168/TP'
'202400433145'
'21/00047 EIASCR'
'21/01348/SCR'
'21/02985/FUL'
'21/03353/FUL'
'21/03553/FUL'
'21/05467/FUL'
'21/05536/FUL'
'22/00601/FUL'
'22/01416/AMM'
'22/01485/PAN'
'22/02054/FUL'
'22/03638/FUL'
'23/0001/EAA'
'23/0004/S36'
'23/00862'
'23/01113/FUL'
'23/0113/FUL'
'23/03113/FUL'
'23/03113/Full Development'
'23/04100/FUL'
'23/05905/FUL'
'231192/PREAPP'
'231480/ESC'
'24/00561/FUL'
'24/00765'
'24/01489/FUL'
'24/01548/FUL'
'24/02621/S36'
'24/0264/FUL'
'24/02641/FUL'
'24/02655/SCOP'
'24/02831/FUL'
'24/03064/SCOP'
'24/05255/S36'
'24/1531/S36'
'240614/DPP'
'240791/S36'
'241074/S36'
'241197-05'
'241197-09'
'241197-12'
'241197/DPP'
'25/00107/FUL'
'25/00307/S36'
'25/00498/S36'
'25/00503/S36'
'25/02382/S36'
'A and B'
'APP/2020/1437'
'APP/2020/1673'
'APP/2021/0936'
'APP/2022/0249'
'APP/2022/0651'
'APP/2022/12

In [10]:
#moving misplaced entities
CODE_TO_ORGANIZATION = {
    'HOLLANDMEY RENEWABLE ENERGY DEVELOPMENT'}

ORGANIZATION_SET.update(CODE_TO_ORGANIZATION)
APPLICATION_CODE_SET.difference_update(CODE_TO_ORGANIZATION)


ORGANIZATION_TO_PLACE = {
'Berwick Bank Wind Farm',
'Kaimes Renewable Energy Park'}

PLACE_SET.update(ORGANIZATION_TO_PLACE)


PERSON_TO_PLACE = ({
'3 Lynn Drive',
'Arran View',
'Glenn Church',
'Gatehead Farm'})

PERSON_TO_ORGANIZATION = ({
'Angus Council',
'Conrad Energy',
'danestone community council',
'Fire Service',
'Fire Scotland',
'Socially Conscious Capital',
'The Beauly Fishing Syndicate',
'the British Trust for Ornithology'})

PLACE_SET.update(PERSON_TO_PLACE)
ORGANIZATION_SET.update(PERSON_TO_ORGANIZATION)


PLACE_TO_ORGANIZATION = ({
    'Department for Energy Security and Net Zero'})

ORGANIZATION_SET.update(PLACE_TO_ORGANIZATION)


In [ ]:
#removing overdetected entities and reassigning misclassified entities
ORGANIZATION_SET.difference_update({
'Badger Trust',
'Bat Conservation Trust',
'BESS',
'Berwick Bank Wind Farm',
'British Trust for Ornithology',
'Buccleuch',
'Buglife Scotland',
'Caithness Bird Club',
'Caithness Bird Clubs',
'Caithness Heath Action Team',
'CALA homes',
'Clean Energy Associates (CEA)',
'Community Council',
'Company',
'Developer',
'Department for Energy Security & Net Zero',
'Drinking Water Quality Regulator for Scotland (DWQR)',
'ECU',
'Electric Power Research Institute',
'Electric Power Research Institute (EPRI)',
'electricity generating company',
'Energy Consents Unit',
'Energy Safety Response Group',
'Energy Workshops',
'European Chemicals Agency',
'Forest and Land Scotland',
'Foundation Scotland',
'Fire Scotland',
'Kaimes Renewable Energy Park',
'National Fire Chief Council',
'National Fire Chiefs Council',
'National Fire Chiefs Council (NFCC)',
'National Fire Protection Association',
'National Fire Safety Services (Ltd)',
'National Fire Safety Services Ltd',
'National Grid',
'National Grid (now NESO)',
'National Grid ESO',
'National Institute for Occupational Safety and Health (NIOSH)',
'Natural Scottish Heritage',
'Nature Scot',
'NatureScot',
'Network Rail',
'NHS Health Scotland',
'Planning Service',
'Scottish Badgers',
'Scottish Environment Protection Agency',
'Scottish Environmental Protection Agency',
'Scottish Environmental Protection Agency (SEPA)',
'Scottish Fire',
'Scottish Fire & Rescue',
'Scottish Fire & Rescue Service',
'Scottish Fire & Rescue Service (SFRS)',
'Scottish Fire and Rescue (SFRS)',
'Scottish Fire and Rescue Service',
'Scottish Fire and Rescue service',
'Scottish Fire and Rescue Service (SFRS)',
'Scottish Fire Brigade',
'Scottish Fire Service',
'Scottish Forestry',
'Scottish Gas',
'Scottish Gas Network',
'Scottish Gas Networks',
'Scottish Government',
'Scottish Government Energy Consents Unit',
"Scottish Government's Energy Consents Unit",
'Scottish Government’s Energy Consents Unit',
'Scottish Hydro Electric Power Distribution (SHEPD)',
'Scottish Hydro Electric Power Distribution plc',
'Scottish Hydro Electric Transmission',
'Scottish Hydro Electric Transmission plc',
'Scottish National Library',
'Scottish Natural Heritage',
'Scottish Planning Policy (SPP)',
'Scottish power',
'Scottish Power',
'Scottish Power Energy Network',
'Scottish Power Energy Networks',
'Scottish Power Energy Networks (SPEN)',
'Scottish Power Renewables',
'Scottish Renewables',
'Scottish Water',
'SEPA',
'SEPA (Scottish Environment Protection Agency)',
'The Applicant',
"the Badger's Trust",
'the local group',
'The Scottish Government',
'The Trust',
'This company',
'UK House of Commons Library',
'UK Infrastructure Planning Inspectorate',
})

In [12]:
APPLICATION_CODE_SET.difference_update({
    'B',
    'A and B',
    'ref C2',
    'ref C5'})

In [13]:
PERSON_SET.difference_update({
'3 Lynn Drive',
'a fireman friend',
'a fraser',
'a local resident and father of young children',
'a resident of the Beauly area',
'a retired engineer',
'All other councillors',
'Angus Council',
'Arran View',
'Conrad Energy',
'Councillor',
'councillors',
'danestone community council',
'Doctor of Physics',
"Eaglesham's Conservation Village status",
'Frankly',
'Gatehead Farm',
'Glenn Church',
"Has the Badger's Trust",
'High School',
'HM Queen Elizabeth the Queen Mother',
'I',
'local councillors',
'local farmers',
'local MP',
'local MSP',
'mine',
'my',
'MSP',
'my daughter',
'My eldest child',
'My eldest son',
'my family',
'my family members',
'my family’s',
'My father',
'my husband',
'my late husband',
'my livestock and pets',
'my mother',
'my mother-in-law',
'My mum',
'my neighbours',
'my son',
'My son',
'my two really young kids',
'My understanding is that National Planning Framework 4 addresses the need for Cumulative Impact to be considered. This development, if approved, would be in a field adjacent to the Zenobe development which has already been approved and close to the other developments mentioned above. Further development in this area will change our environment out of all recognition.',
'My wife',
'my wife and I',
'my young children',
'my youngest child',
'myself',
'Our Client',
'our friends',
'Our Local MP',
'Our two young children',
'Phillips',
'Police Scotland',
'Queen Mother',
'S SIMEC Atlantis Energy (SAE) Renewables',
'Senior Pastor',
'several pets',
'Socially Conscious Capital',
'The Beauly Fishing Syndicate',
'the British Trust for Ornithology',
'The local MP',
'the original farmer',
'the three councillors',
'The two community councils',
})

In [ ]:
PLACE_SET.difference_update({
'a large quarry',
'a lively area',
'a plethora of pylons',
'a police station',
'a pond',
'a power station and dam',
'a productive salmon river',
'a single fence',
'a small village community',
'a substation',
'a very nice area',
'a very nice village',
'a water course',
'a wedding venue',
'a wind farm',
'an extremely busy roundabout',
'Ancient Woodland',
'childrens play park',
'children’s play park',
'Community Centre',
'conservation village',
'Conservation Village',
'Conservation village',
'Department for Energy Security and Net Zero',
'doctors',
'doctors surgery',
'Doctors Surgery',
'European landscape convention',
'Fig. 1',
'Fig. 2',
'former coal plant',
'Greenbelt',
'Green Belt',
'Grid Stability Facility',
'grounds',
'historic sites',
'local children’s primary school',
'local nature reserve',
'local nursing home',
'local primary and nursery schools',
'local primary school',
'local residential area',
'motorway',
'my land',
'my neighbourhood',
'my own croft',
'my property and horse stables',
'nearby river',
'nearby school playing fields',
'nearby watercourses',
'nearby woodland',
'nearest an and e',
'nearest fire station',
'new housing development',
'new road',
'nursery',
'old coal store area',
'old power station site',
'our beautiful village',
'our small single track road',
'our village',
'our villiage',
'overlooking properties',
'play park',
'playing fields',
'playpark',
'Primary and Secondary School',
'primary school',
'Primary School',
'railway line',
'residential houses',
'residential housing',
'rhe wind farm',
'Roman frontier',
'roundabout',
'Scrapdealers',
'Secondary School',
'single-track road',
'Site of Importance for Nature Conservation',
'Site of Special Scientific Interest',
'Site of Special Scientific Interest (SSSI)',
'Site of Special Scientific Interest which is the moss',
'Site of Specific Scientific Interest',
'the coast',
'the conservation village',
'the environment round here',
'the ford',
'the hill',
'the lochs',
'the moor',
'the moors',
'the old mills',
'the proposed site',
'the quarry',
'the river',
'the school',
'the site',
'the track',
'the village',
'the woods next to the site',
'this part of the country',
'the back of a housing estate',
'the back of one of the poorest housing estates in the area',
'this proposed site',
'SEPA flood risk mapping',
'the burn',
'this village',
'UNESCO site',
'Village',
'village',
'wall',
'water works',
'woodland',
'Woodlands'})


In [15]:
approved_candidate_sets = {
    category: sorted(values, key=str.casefold)
    for category, values in candidate_sets.items()}

with open("approved_masking_candidates.json", "w", encoding="utf-8") as file:
    json.dump(approved_candidate_sets, file, ensure_ascii=False, indent=2)

**MASKING**

In [72]:
INPUT_CSV = "SCOTBESS_DATASET_MASKED.csv"
APPROVED_SETS_JSON = "approved_masking_candidates.json"
OUTPUT_CSV = "SCOTBESS_DATASET_MASKED_REVIEWED_FINAL.csv"

df = pd.read_csv(INPUT_CSV)

In [ ]:
DIRECT_CATEGORIES = ["EMAIL", "PHONE", "LINK"]

DIRECT_PLACEHOLDERS = {
    "EMAIL": "[EMAIL]",
    "PHONE": "[PHONE]",
    "LINK": "[LINK]"}


def parse_entities(raw_output):
    if pd.isna(raw_output) or not str(raw_output).strip():
        return {}

    try:
        entities = (
            json.loads(raw_output)
            if isinstance(raw_output, str)
            else raw_output
        )
    except (json.JSONDecodeError, TypeError):
        return {}

    return entities if isinstance(entities, dict) else {}


def mask_direct_entities(row):
    """Mask only EMAIL, PHONE and LINK detected in this particular row."""

    if pd.isna(row["text"]):
        return row["text"]

    text = str(row["text"])
    entities = parse_entities(row["llm_entities_json"])
    matches = []

    for category in DIRECT_CATEGORIES:
        values = entities.get(category, [])

        if not isinstance(values, list):
            continue

        for raw_value in values:
            value = str(raw_value).strip()

            if not value:
                continue

            #Exact literal matching because the LLM was required to return values character-for-character from this row
            start = 0

            while True:
                position = text.find(value, start)

                if position == -1:
                    break

                matches.append({
                    "start": position,
                    "end": position + len(value),
                    "category": category,
                })

                start = position + len(value)

    #prefering longer spans if candidates overlap
    matches.sort(
        key=lambda item: (
            -(item["end"] - item["start"]),
            item["start"]))

    selected = []

    for candidate in matches:
        overlaps = any(
            candidate["start"] < existing["end"]
            and candidate["end"] > existing["start"]
            for existing in selected)

        if not overlaps:
            selected.append(candidate)

    result = text

    for match in sorted(
        selected,
        key=lambda item: item["start"],
        reverse=True):
        result = (
            result[:match["start"]]
            + DIRECT_PLACEHOLDERS[match["category"]]
            + result[match["end"]:])

    return result

In [ ]:
df["direct_masked_text"] = df.apply(mask_direct_entities, axis=1)

In [ ]:
#utuilizing regex for additional masking of emails, phone numbers and links detection, as the llm failed to detect some
def mask_contacts(text, print_masked=True):
    masked_items = []

    def repl_email(match):
        masked_items.append(("EMAIL", match.group(0)))
        return "[EMAIL]"

    def repl_phone(match):
        masked_items.append(("PHONE", match.group(0)))
        return "[PHONE]"

    #Emails
    email_pattern = r"""
    \b
    [A-Z0-9._%+-]+
    @
    [A-Z0-9.-]+
    \.
    [A-Z]{2,}
    \b
    """

    text = re.sub(email_pattern, repl_email, text, flags=re.IGNORECASE | re.VERBOSE)

    #UK-ish phone numbers
    phone_pattern = r"""
    (?<!\w)
    (?:
        (?:\+44\s?(?:\(0\)\s?)?|0)
        7\d{3}[\s\-]?\d{3}[\s\-]?\d{3}
        |
        \(0\d{2,5}\)[\s\-]?\d{3,4}[\s\-]?\d{3,4}
        |
        (?:\+44\s?(?:\(0\)\s?)?|0)
        \d{2,5}[\s\-]?\d{3,4}[\s\-]?\d{3,4}
    )
    (?!\w)
    """

    text = re.sub(phone_pattern, repl_phone, text, flags=re.VERBOSE)

    if print_masked and masked_items:
        print(f"\nMASKED CONTACTS:")
        for kind, item in masked_items:
            print(f"{kind}: {item}")
        print("-" * 80)

    return text


In [ ]:
def mask_links(text, print_masked=True):
    masked_items = []

    def repl(match):
        masked_items.append(match.group(0))
        return "[LINK]"

    link_pattern = r"""
    (?:
        https?://[^\s<>()]+
        |
        www\.[^\s<>()]+
    )
    """

    text = re.sub(link_pattern, repl, text, flags=re.IGNORECASE | re.VERBOSE)

    if print_masked and masked_items:
        print(f"\nMASKED LINKS:")
        for item in masked_items:
            print(item)
        print("-" * 80)

    return text

In [77]:
def mask_references(text, print_masked=True):
    masked_items = []

    def repl(match):
        masked_items.append(match.group(0))
        return "[APPLICATION_CODE]"

    reference_pattern = r"""
    (?<![A-Z0-9])
    (?:
        ECU\d{5,}
        |EC\d{5,}
        |APP/\d{4}/\d{3,6}
        |ENQ/\d{4}/\d{3,6}
        |LIVE/\d{3,6}/[A-Z]+/\d{2,4}
        |P/\d{2}/\d{3,6}(?:/[A-Z]+)?
        |\d{2}/\d{3,6}/[A-Z0-9]+
        |\d{3,6}/[A-Z]+/\d{2,4}
    )
    (?![A-Z0-9])
    """

    text = re.sub(reference_pattern, repl, text, flags=re.IGNORECASE | re.VERBOSE)

    if print_masked and masked_items:
        print(f"\nMASKED REFERENCES:")
        for item in masked_items:
            print(item)
        print("-" * 80)

    return text

In [78]:
def pii_masking_pipeline(text, print_masked=True):
    text = mask_contacts(text, print_masked=print_masked)
    text = mask_links(text, print_masked=print_masked)

    return text

In [ ]:
df["regex_masked_text"] = df["direct_masked_text"].apply(lambda text: pii_masking_pipeline(text, print_masked=True))


MASKED LINKS:
https://www.nasa.gov/sites/default/files/atoms/files/nabw20_fire_gas_char_studies_liion_cells_batt_djuarez-
--------------------------------------------------------------------------------

MASKED LINKS:
www.ssen.co.uk
--------------------------------------------------------------------------------

MASKED LINKS:
https://hansard.parliament.uk/Commons/2022-09-07/debates/FB0D6FE6-CF3E-
--------------------------------------------------------------------------------

MASKED LINKS:
https://online.aberdeenshire.gov.uk/smrpub/master/detail.aspx?Authority=ASH&refno=NO87NW0
--------------------------------------------------------------------------------

MASKED LINKS:
https://www.gov.uk/government/publications/grid-scale-electrical-energystorage-systems-health-
https://www.gov.uk/government/publications/grid-scale-electrical-energystorage-systems-health-
--------------------------------------------------------------------------------

MASKED CONTACTS:
PHONE: 000000211
----------

In [ ]:
#masking of the reviewed categories
REVIEWED_CATEGORIES = [
    "APPLICATION_CODE",
    "PERSON",
    "ORGANIZATION",
    "PLACE"]

with open(APPROVED_SETS_JSON, "r", encoding="utf-8") as file:
    loaded_sets = json.load(file)

approved_sets = {
category: {
        str(value).strip()
        for value in loaded_sets.get(category, [])
        if str(value).strip()}
    for category in REVIEWED_CATEGORIES}

for category, values in approved_sets.items():
    print(f"{category}: {len(values)} reviewed values")

APPLICATION_CODE: 144 reviewed values
PERSON: 142 reviewed values
ORGANIZATION: 264 reviewed values
PLACE: 1399 reviewed values


In [ ]:
REVIEWED_PLACEHOLDERS = {
    "APPLICATION_CODE": "[APPLICATION_CODE]",
    "PERSON": "[PERSON]",
    "ORGANIZATION": "[ORGANIZATION]",
    "PLACE": "[PLACE]"}


def compile_value_pattern(values):
    values = {
        str(value).strip()
        for value in values
        if str(value).strip()}

    if not values:
        return None

    ordered_values = sorted(values, key=lambda value: (-len(value), value.casefold()))

    return re.compile(
        r"(?<!\w)(?:"
        + "|".join(re.escape(value) for value in ordered_values)
        + r")(?![\w'’])")


reviewed_patterns = {category: compile_value_pattern(approved_sets[category]) for category in REVIEWED_CATEGORIES}


def mask_reviewed_entities(text):
    if pd.isna(text):
        return text

    text = str(text)
    matches = []

    for category, pattern in reviewed_patterns.items():
        if pattern is None:
            continue

        for match in pattern.finditer(text):
            matches.append({
                "start": match.start(),
                "end": match.end(),
                "category": category})

    #prefers the longest value, such as an organization name, over a shorter place contained inside it
    matches.sort(
        key=lambda item: (
            -(item["end"] - item["start"]),
            item["start"]))

    selected = []

    for candidate in matches:
        overlaps = any(
            candidate["start"] < existing["end"]
            and candidate["end"] > existing["start"]
            for existing in selected)

        if not overlaps:
            selected.append(candidate)

    result = text

    for match in sorted(
        selected,
        key=lambda item: item["start"],
        reverse=True,):
        result = (
            result[:match["start"]]
            + REVIEWED_PLACEHOLDERS[match["category"]]
            + result[match["end"]:])

    return result

In [82]:
df["reviewed_masked_text"] = (df["regex_masked_text"].apply(mask_reviewed_entities))

In [83]:
#fallback regex masking for application reference codes.
def mask_references(text, print_masked=True):
    masked_items = []

    def repl(match):
        masked_items.append(match.group(0))
        return "[APPLICATION_CODE]"

    reference_pattern = r"""
    (?<![A-Z0-9])
    (?:
        ECU\d{5,}
        |EC\d{5,}
        |APP/\d{4}/\d{3,6}
        |ENQ/\d{4}/\d{3,6}
        |LIVE/\d{3,6}/[A-Z]+/\d{2,4}
        |P/\d{2}/\d{3,6}(?:/[A-Z]+)?
        |\d{2}/\d{3,6}/[A-Z0-9]+
        |\d{3,6}/[A-Z]+/\d{2,4}
    )
    (?![A-Z0-9])
    """

    text = re.sub(reference_pattern, repl, text, flags=re.IGNORECASE | re.VERBOSE)

    if print_masked and masked_items:
        print(f"\nMASKED REFERENCES:")
        for item in masked_items:
            print(item)
        print("-" * 80)

    return text

In [84]:
def pii_masking_pipeline(text, print_masked=True):
    text = mask_references(text, print_masked=print_masked)

    return text

In [85]:
df["final_masked_text"] = df["reviewed_masked_text"].apply(
    lambda text: pii_masking_pipeline(text, print_masked=True))


MASKED REFERENCES:
21/05467/FULby
--------------------------------------------------------------------------------

MASKED REFERENCES:
ECU00005190
13/02682/FUL25
--------------------------------------------------------------------------------

MASKED REFERENCES:
ECU00003458
--------------------------------------------------------------------------------


In [ ]:
#for inspection
display(
    df[
        [
            "text",
            "masked_text",          #old LLM-generated text, audit only
            "direct_masked_text",   #row-specific LLM EMAIL/PHONE/LINK
            "regex_masked_text",    #regex contacts/links/references
            "reviewed_masked_text",    #reviewed entity-set result
            "final_masked_text", # with fallback for application reference codes applied
            "llm_entities_json"
        ]].head(2))


OUTPUT_COLUMNS = [
    "project",
    "filename",
    "source",
    "final_masked_text"]

df[OUTPUT_COLUMNS].to_csv(OUTPUT_CSV, index=False, encoding="utf-8",)
df.to_excel("SCOTBESS_DATASET_MASKED_AND_REVIEWED_FINAL.xlsx", index=False)


,text,masked_text,direct_masked_text,regex_masked_text,reviewed_masked_text,final_masked_text,llm_entities_json
0,The ground on which the proposed energy site i...,The ground on which the proposed energy site i...,The ground on which the proposed energy site i...,The ground on which the proposed energy site i...,The ground on which the proposed energy site i...,The ground on which the proposed energy site i...,"{""EMAIL"": [], ""PHONE"": [], ""LINK"": [], ""APPLIC..."
1,We have serious reservations that there has no...,We have serious reservations that there has no...,We have serious reservations that there has no...,We have serious reservations that there has no...,We have serious reservations that there has no...,We have serious reservations that there has no...,"{""EMAIL"": [], ""PHONE"": [], ""LINK"": [""https://w..."


In [87]:
df.to_excel("SCOTBESS_DATASET_MASKED_AND_REVIEWED_FINAL.xlsx", index=False)